# 05 — Ablation study

Zero-ablate the top-K heads. Refusal rate should collapse on held-out toxic prompts; perplexity should barely move.


In [ ]:
from safety_circuits.config import MODELS, REPO_ROOT, RESULTS_DIR
from safety_circuits.models import load_model
from safety_circuits.ablation import HeadRef, evaluate_ablation, perplexity
from safety_circuits.data import load_jsonl
import pandas as pd


In [ ]:
loaded = load_model(MODELS['qwen'])
agg = pd.read_csv(RESULTS_DIR / 'qwen_patch_z.csv')
top = agg[agg.component == 'z'].head(10)
heads = [HeadRef(int(r.layer), int(r.head)) for r in top.itertuples()]
heads


In [ ]:
pairs = load_jsonl(REPO_ROOT / 'data' / 'processed' / 'pairs.jsonl')[20:60]  # held-out
eval_prompts = [r['harm']['text'] for r in pairs]
report = evaluate_ablation(loaded, heads, eval_prompts, mode='zero')
print(report)


### Capability check — did we break the model?


In [ ]:
wikitext = [
    'The quick brown fox jumps over the lazy dog.',
    'In a hole in the ground there lived a hobbit.',
    'It was the best of times, it was the worst of times.',
]
print('clean ppl:', perplexity(loaded, wikitext))


### Replicate on Phi-3


In [ ]:
# loaded_phi = load_model(MODELS['phi3'])
# repeat patching + ablation as above.
